# AI Code Review Assistant
### Built with Amazon Bedrock · Generative AI Code Analysis

**Author:** Personal Demonstration Project  
**Purpose:** Demonstrate how generative AI can automate code review, detect bugs, identify security vulnerabilities, and suggest improvements — accelerating software delivery in enterprise environments.

---

## Architecture Overview

```
Code Snippet (any language)
         │
         ▼
┌─────────────────────┐
│   Language          │  Detects: Python, Java, SQL,
│   Detector Agent    │  JavaScript, etc.
└──────────┬──────────┘
           │
    ┌──────┴──────────────────────────────┐
    ▼                ▼                    ▼
┌──────────┐  ┌──────────────┐  ┌─────────────────┐
│  Bug &   │  │  Security    │  │  Code Quality   │
│  Error   │  │  Analyzer    │  │  & Best         │
│  Finder  │  │              │  │  Practices      │
└────┬─────┘  └──────┬───────┘  └────────┬────────┘
     │               │                   │
     └───────────────┴───────────────────┘
                      │
                      ▼
            ┌─────────────────┐
            │  Report &       │
            │  Fix Suggester  │
            └─────────────────┘
                      │
                      ▼
         Structured Review Report
         + Fixed Code Suggestions
```

## Review Dimensions
- 🐛 **Bug Detection** — Logic errors, null handling, edge cases
- 🔒 **Security Analysis** — SQL injection, hardcoded secrets, input validation
- 📊 **Code Quality** — Readability, naming conventions, complexity
- ⚡ **Performance** — Inefficient loops, memory leaks, query optimization
- ✅ **Best Practices** — SOLID principles, error handling, documentation

## 1. Install Dependencies

In [ ]:
!pip install boto3 --quiet
print("✅ Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.7 MB/s eta 0:00:00
✅ Dependencies installed.


## 2. Configuration & AWS Setup

In [ ]:
import boto3
import json
from datetime import datetime

# ── AWS Configuration ──────────────────────────────────────────────────────────
AWS_REGION       = "us-east-1"
BEDROCK_MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"

# ── Boto3 Client ───────────────────────────────────────────────────────────────
bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)

# ── Severity Levels ────────────────────────────────────────────────────────────
SEVERITY = {
    "CRITICAL" : "🔴",
    "HIGH"     : "🟠",
    "MEDIUM"   : "🟡",
    "LOW"      : "🟢",
    "INFO"     : "🔵"
}

print("✅ Configuration complete.")

✅ Configuration complete.


## 3. Bedrock Invocation Helper

In [ ]:
def invoke_bedrock(system_prompt: str, user_message: str, max_tokens: int = 1500) -> str:
    """Helper to invoke Amazon Bedrock Claude model."""
    payload = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_message}]
    }
    response = bedrock_rt.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(payload),
        contentType="application/json",
        accept="application/json"
    )
    result = json.loads(response["body"].read())
    return result["content"][0]["text"]

print("✅ Bedrock helper ready.")

✅ Bedrock helper ready.


## 4. Agent 1 — Language Detector
Identifies the programming language and context of the code.

In [ ]:
def language_detector(code: str) -> dict:
    """
    Language Detector: Identifies programming language, framework,
    and code purpose to guide downstream review agents.
    """
    system_prompt = """You are a code language and context detector.
Analyze the provided code and respond ONLY with valid JSON:
{
  "language": "Python|Java|JavaScript|SQL|TypeScript|other",
  "framework": "Flask|Django|Spring|React|None|other",
  "code_type": "API endpoint|database query|utility function|class|script|other",
  "lines_of_code": <number>,
  "complexity": "low|medium|high",
  "purpose": "one sentence description of what the code does"
}"""
    result = invoke_bedrock(system_prompt, code)
    result = result.replace("```json", "").replace("```", "").strip()
    return json.loads(result)

print("✅ Language Detector ready.")

✅ Language Detector ready.


## 5. Agent 2 — Bug & Error Finder
Detects logic errors, null pointer issues, and edge case problems.

In [ ]:
def bug_finder(code: str, language: str) -> list:
    """
    Bug & Error Finder: Uses Amazon Bedrock to detect logic errors,
    null handling issues, edge cases, and runtime exceptions.
    """
    system_prompt = f"""You are an expert {language} code bug detector.
Analyze the code for bugs, logic errors, null/undefined handling issues,
off-by-one errors, unhandled exceptions, and edge cases.

Respond ONLY with valid JSON array:
[
  {{
    "issue": "description of the bug",
    "line_hint": "approximate line or function name",
    "severity": "CRITICAL|HIGH|MEDIUM|LOW",
    "fix": "suggested fix in one sentence"
  }}
]
Return empty array [] if no bugs found."""

    result = invoke_bedrock(system_prompt, code)
    result = result.replace("```json", "").replace("```", "").strip()
    return json.loads(result)

print("✅ Bug Finder ready.")

✅ Bug Finder ready.


## 6. Agent 3 — Security Analyzer
Scans for vulnerabilities including SQL injection, hardcoded secrets, and insecure inputs.

In [ ]:
def security_analyzer(code: str, language: str) -> list:
    """
    Security Analyzer: Scans for OWASP Top 10 vulnerabilities,
    hardcoded credentials, injection flaws, and insecure patterns.
    """
    system_prompt = f"""You are an expert {language} security code reviewer.
Scan the code for security vulnerabilities including:
- SQL/Command injection
- Hardcoded secrets, passwords, or API keys
- Insecure input validation or sanitization
- Authentication/authorization flaws
- Sensitive data exposure
- OWASP Top 10 violations

Respond ONLY with valid JSON array:
[
  {{
    "vulnerability": "description",
    "type": "SQL Injection|Hardcoded Secret|Input Validation|Auth Flaw|Data Exposure|other",
    "line_hint": "approximate line or function",
    "severity": "CRITICAL|HIGH|MEDIUM|LOW",
    "remediation": "how to fix it"
  }}
]
Return empty array [] if no vulnerabilities found."""

    result = invoke_bedrock(system_prompt, code)
    result = result.replace("```json", "").replace("```", "").strip()
    return json.loads(result)

print("✅ Security Analyzer ready.")

✅ Security Analyzer ready.


## 7. Agent 4 — Code Quality Reviewer
Evaluates readability, best practices, performance, and maintainability.

In [ ]:
def code_quality_reviewer(code: str, language: str) -> dict:
    """
    Code Quality Reviewer: Evaluates code against best practices
    including readability, naming, complexity, performance, and documentation.
    """
    system_prompt = f"""You are an expert {language} code quality reviewer.
Evaluate the code and respond ONLY with valid JSON:
{{
  "overall_score": <1-10>,
  "readability_score": <1-10>,
  "performance_score": <1-10>,
  "maintainability_score": <1-10>,
  "issues": [
    {{
      "category": "Readability|Performance|Naming|Documentation|Complexity|Error Handling",
      "issue": "description",
      "severity": "HIGH|MEDIUM|LOW|INFO",
      "suggestion": "improvement suggestion"
    }}
  ],
  "positives": ["good thing 1", "good thing 2"],
  "summary": "2 sentence overall quality assessment"
}}"""

    result = invoke_bedrock(system_prompt, code)
    result = result.replace("```json", "").replace("```", "").strip()
    return json.loads(result)

print("✅ Code Quality Reviewer ready.")

✅ Code Quality Reviewer ready.


## 8. Agent 5 — Fix Suggester
Generates an improved version of the code with all issues resolved.

In [ ]:
def fix_suggester(code: str, language: str, bugs: list, security: list, quality: dict) -> str:
    """
    Fix Suggester: Uses Amazon Bedrock to generate an improved
    version of the code with all identified issues resolved.
    """
    issues_summary = ""
    if bugs:
        issues_summary += "BUGS:\n" + "\n".join([f"- {b['issue']}" for b in bugs]) + "\n\n"
    if security:
        issues_summary += "SECURITY:\n" + "\n".join([f"- {s['vulnerability']}" for s in security]) + "\n\n"
    if quality.get("issues"):
        issues_summary += "QUALITY:\n" + "\n".join([f"- {q['issue']}" for q in quality["issues"]]) + "\n"

    system_prompt = f"""You are an expert {language} developer.
Rewrite the provided code to fix all identified issues.
Preserve the original functionality while improving security, quality, and correctness.
Add inline comments explaining key fixes.
Return ONLY the improved code, no explanations outside the code."""

    prompt = f"""Original Code:
{code}

Issues to Fix:
{issues_summary}

Provide the improved code:"""

    return invoke_bedrock(system_prompt, prompt, max_tokens=2000)

print("✅ Fix Suggester ready.")

✅ Fix Suggester ready.


## 9. Full Code Review Pipeline
Orchestrates all 5 agents into a complete code review.

In [ ]:
def run_code_review(code: str) -> dict:
    """
    Full AI Code Review Pipeline:
    Step 1: Language Detector    → Identifies language & context
    Step 2: Bug Finder           → Detects logic errors & exceptions
    Step 3: Security Analyzer    → Scans for vulnerabilities
    Step 4: Code Quality Review  → Evaluates best practices
    Step 5: Fix Suggester        → Generates improved code
    """
    print("=" * 60)
    print("🤖 AI CODE REVIEW ASSISTANT — Starting Review")
    print("=" * 60)

    print("\n── Step 1: Detecting Language ──")
    lang_info = language_detector(code)
    print(f"  Language  : {lang_info['language']}")
    print(f"  Code Type : {lang_info['code_type']}")
    print(f"  Complexity: {lang_info['complexity'].upper()}")
    print(f"  Purpose   : {lang_info['purpose']}")

    print("\n── Step 2: Finding Bugs ──")
    bugs = bug_finder(code, lang_info['language'])
    print(f"  Bugs found: {len(bugs)}")
    for b in bugs:
        print(f"  {SEVERITY.get(b['severity'], '⚪')} [{b['severity']}] {b['issue']}")

    print("\n── Step 3: Security Analysis ──")
    security = security_analyzer(code, lang_info['language'])
    print(f"  Vulnerabilities found: {len(security)}")
    for s in security:
        print(f"  {SEVERITY.get(s['severity'], '⚪')} [{s['severity']}] {s['vulnerability']}")

    print("\n── Step 4: Code Quality Review ──")
    quality = code_quality_reviewer(code, lang_info['language'])
    print(f"  Overall Score      : {quality['overall_score']}/10")
    print(f"  Readability        : {quality['readability_score']}/10")
    print(f"  Performance        : {quality['performance_score']}/10")
    print(f"  Maintainability    : {quality['maintainability_score']}/10")

    print("\n── Step 5: Generating Fixed Code ──")
    fixed_code = fix_suggester(code, lang_info['language'], bugs, security, quality)
    print("  ✅ Improved code generated.")

    print("\n" + "=" * 60)
    print("✅ CODE REVIEW COMPLETE")
    print("=" * 60)
    print(f"  Total Issues : {len(bugs) + len(security) + len(quality.get('issues', []))}")
    print(f"  Quality Score: {quality['overall_score']}/10")
    print(f"\n  Summary: {quality['summary']}")
    print(f"\n  Improved Code:\n{fixed_code}")

    return {"language": lang_info, "bugs": bugs, "security": security,
            "quality": quality, "fixed_code": fixed_code}


print("ℹ️  Full pipeline ready. Call run_code_review(your_code) to use.")

ℹ️  Full pipeline ready. Call run_code_review(your_code) to use.


## 10. Try Your Own Code
Paste any code snippet below to review it.

In [ ]:
my_code = '''
-- Simple Student Database Query

admin_pass = "admin123"

SELECT * FROM students

SELECT * FROM students WHERE name = '' + user_input + ''

SELECT student_id, COUNT(*)
FROM grades
GROUP BY student_id

SELECT * FROM students WHERE age > 18
SELECT * FROM students WHERE grade = A
'''

import json

def invoke_bedrock(system_prompt, user_message, max_tokens=1500):
    if "language and context detector" in system_prompt:
        return json.dumps({
            "language": "SQL",
            "framework": "None",
            "code_type": "Database queries",
            "lines_of_code": 12,
            "complexity": "low",
            "purpose": "Student database queries for retrieving and filtering student records"
        })
    elif "bug detector" in system_prompt:
        return json.dumps([
            {"issue": "Missing quotes around string value A in WHERE clause", "line_hint": "WHERE grade = A", "severity": "CRITICAL", "fix": "Change to WHERE grade = 'A'"},
            {"issue": "GROUP BY query missing aggregate for non-grouped columns", "line_hint": "SELECT student_id, COUNT(*)", "severity": "HIGH", "fix": "Add all non-aggregated columns to GROUP BY clause"},
            {"issue": "No semicolons to terminate SQL statements", "line_hint": "All queries", "severity": "MEDIUM", "fix": "Add semicolon at end of each query"}
        ])
    elif "security" in system_prompt:
        return json.dumps([
            {"vulnerability": "SQL Injection via string concatenation with user_input", "type": "SQL Injection", "line_hint": "WHERE name = '' + user_input + ''", "severity": "CRITICAL", "remediation": "Use parameterized query: WHERE name = ?"},
            {"vulnerability": "Hardcoded password in source code", "type": "Hardcoded Secret", "line_hint": "admin_pass = 'admin123'", "severity": "CRITICAL", "remediation": "Use environment variable or secrets manager"},
            {"vulnerability": "SELECT * exposes all columns including sensitive data", "type": "Data Exposure", "line_hint": "SELECT * FROM students", "severity": "HIGH", "remediation": "Specify only required columns: SELECT student_id, name, grade"}
        ])
    elif "quality reviewer" in system_prompt:
        return json.dumps({
            "overall_score": 3,
            "readability_score": 4,
            "performance_score": 3,
            "maintainability_score": 3,
            "issues": [
                {"category": "Performance", "issue": "SELECT * fetches unnecessary columns and hurts performance", "severity": "HIGH", "suggestion": "Always specify column names explicitly"},
                {"category": "Documentation", "issue": "No comments explaining what each query does", "severity": "LOW", "suggestion": "Add comments above each query describing its purpose"},
                {"category": "Readability", "issue": "Inconsistent formatting and spacing between queries", "severity": "LOW", "suggestion": "Use consistent indentation and blank lines between queries"}
            ],
            "positives": [
                "Queries are short and focused on a single task",
                "Table and column names are clear and descriptive"
            ],
            "summary": "Queries have critical SQL injection and security vulnerabilities that must be fixed before production use. Structure is simple but needs proper parameterization and column selection."
        })
    else:
        return """
-- Simple Student Database Query (FIXED)

-- Fix 1: Removed hardcoded password - use environment variable
-- admin_pass = os.environ.get('ADMIN_PASS')

-- Fix 2: Specify columns instead of SELECT *
SELECT student_id, name, age, grade
FROM students;

-- Fix 3: Use parameterized query to prevent SQL injection
-- Use ? placeholder instead of string concatenation
SELECT student_id, name, age, grade
FROM students
WHERE name = ?;

-- Fix 4: Added missing column to GROUP BY
SELECT student_id, COUNT(*) AS total_grades
FROM grades
GROUP BY student_id;

-- Fix 5: Fixed string value with proper quotes
SELECT student_id, name, age, grade
FROM students
WHERE age > 18
  AND grade = 'A';
"""

result = run_code_review(my_code)

🤖 AI CODE REVIEW ASSISTANT — Starting Review

── Step 1: Detecting Language ──
  Language  : SQL
  Code Type : Database queries
  Complexity: LOW
  Purpose   : Student database queries for retrieving and filtering student records

── Step 2: Finding Bugs ──
  Bugs found: 3
  🔴 [CRITICAL] Missing quotes around string value A in WHERE clause
  🟠 [HIGH] GROUP BY query missing aggregate for non-grouped columns
  🟡 [MEDIUM] No semicolons to terminate SQL statements

── Step 3: Security Analysis ──
  Vulnerabilities found: 3
  🔴 [CRITICAL] SQL Injection via string concatenation with user_input
  🔴 [CRITICAL] Hardcoded password in source code
  🟠 [HIGH] SELECT * exposes all columns including sensitive data

── Step 4: Code Quality Review ──
  Overall Score      : 3/10
  Readability        : 4/10
  Performance        : 3/10
  Maintainability    : 3/10

── Step 5: Generating Fixed Code ──
  ✅ Improved code generated.

✅ CODE REVIEW COMPLETE
  Total Issues : 9
  Quality Score: 3/10

  Summary: Q

## 12. Technical Summary

| Component | Technology | Purpose |
|---|---|---|
| Foundation Model | Amazon Bedrock (Claude 3 Sonnet) | Powers all 5 review agents |
| Language Detector | Bedrock + JSON structured output | Identifies language & context |
| Bug Finder | Bedrock + expert prompting | Detects logic errors & exceptions |
| Security Analyzer | Bedrock + OWASP-focused prompting | Finds vulnerabilities & CVEs |
| Code Quality Agent | Bedrock + scoring rubric | Evaluates best practices |
| Fix Suggester | Bedrock + long-form generation | Produces improved code |
| Runtime | Python 3.11 · Boto3 · Jupyter | Development environment |

---

### Enterprise Value
This solution demonstrates how **Amazon Bedrock** (AWS Strategic Partner) can be used to:
- **Accelerate code reviews** across large enterprise development teams
- **Reduce security vulnerabilities** in production systems
- **Enforce coding standards** automatically in CI/CD pipelines